## 1. Importação de bibliotecas

In [1]:
!pip install transformers torch accelerate -q
import random
import csv
import pandas as pd
import numpy as np
import torch
import random
from transformers import AutoModelForCausalLM, AutoTokenizer # AutoTokenizer e o AutoModelForCausalLM automatizam o processo de importação do modelo, puxando os dados e informações necessárias diretamente do Hugging Face

## 2. Geração de dados
Os dados foram gerados tentando manter uma proporção 33%/33%/33% com base no tipo de imóvel.

In [2]:
"""
Gerador de base de dados sintética para o problema de análise energética.
Gera apenas as colunas de entrada (features), com consumo_kwh correlacionado
de forma realista ao tipo de imóvel, quantidade de equipamentos e horas de
alto consumo, em vez de valores puramente aleatórios.
"""

import random
import csv

TIPOS_IMOVEL = ["Casa", "Apartamento", "Comercial"]
TARIFA_KWH = 0.75


def gerar_registro(id_cliente):
    tipo_imovel = random.choice(TIPOS_IMOVEL)
    quantidade_equipamentos = random.randint(1, 25)
    horas_alto_consumo = round(random.uniform(0, 12), 1)
    uso_horario_pico = random.random() < 0.5

    # Consumo base varia conforme tipo de imóvel
    base_por_tipo = {
        "Casa": 250,
        "Apartamento": 150,
        "Comercial": 500,
    }[tipo_imovel]

    # Consumo cresce com equipamentos e horas de alto consumo, com ruído aleatório
    consumo_kwh = (
        base_por_tipo
        + quantidade_equipamentos * random.uniform(8, 15)
        + horas_alto_consumo * random.uniform(10, 20)
        + (50 if uso_horario_pico else 0)
        + random.gauss(0, 30)  # ruído
    )
    consumo_kwh = max(20, round(consumo_kwh, 1))

    return {
        "id_cliente": id_cliente,
        "consumo_kwh": consumo_kwh,
        "uso_horario_pico": uso_horario_pico,
        "quantidade_equipamentos": quantidade_equipamentos,
        "tipo_imovel": tipo_imovel,
        "horas_alto_consumo": horas_alto_consumo,
    }


def gerar_base(n_registros=5000, caminho_saida="base_energetica.csv"):
    registros = [gerar_registro(id_cliente=i) for i in range(1, n_registros + 1)]

    with open(caminho_saida, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=registros[0].keys())
        writer.writeheader()
        writer.writerows(registros)

    print(f"Base gerada com {n_registros} registros em '{caminho_saida}'")
    return registros


if __name__ == "__main__":
    gerar_base(n_registros=5000, caminho_saida="base_energetica.csv")

Base gerada com 5000 registros em 'base_energetica.csv'


In [3]:
df = pd.read_csv("/content/base_energetica.csv")

## 3. Definição de categorias

In [4]:
"""
1. Rotula a base sintética criando um índice de ineficiência energética
   e cortando-o em tercis (Eficiente / Moderado / Ineficiente).
2. Treina e compara três classificadores: Regressão Logística, Random Forest
   e Árvore de Decisão.
3. Salva o melhor modelo (pipeline completo com pré-processamento).
"""

CAMINHO_ENTRADA = "base_energetica.csv"
CAMINHO_SAIDA = "base_energetica_rotulada.csv"

COLUNAS_NUMERICAS = ["consumo_kwh", "quantidade_equipamentos", "horas_alto_consumo"]
COLUNAS_CATEGORICAS = ["tipo_imovel"]
COLUNA_BOOLEANA = "uso_horario_pico"

# Consumo médio de referência por tipo de imóvel (mesma lógica usada na geração da base)
CONSUMO_BASE_POR_TIPO = {
    "Casa": 250,
    "Apartamento": 150,
    "Comercial": 500,
}


def calcular_indice_ineficiencia(df):
    """Índice composto (0 a 1) combinando as 3 variáveis de entrada."""
    consumo_relativo = df["consumo_kwh"] / df["tipo_imovel"].map(CONSUMO_BASE_POR_TIPO)
    consumo_norm = (consumo_relativo / consumo_relativo.max()).clip(0, 1)
    equip_norm = (df["quantidade_equipamentos"] / df["quantidade_equipamentos"].max()).clip(0, 1)
    horas_norm = (df["horas_alto_consumo"] / df["horas_alto_consumo"].max()).clip(0, 1)
    pico_norm = df["uso_horario_pico"].astype(int)

    indice = (
        0.40 * consumo_norm
        + 0.25 * pico_norm
        + 0.20 * equip_norm
        + 0.15 * horas_norm
    )
    return indice


def rotular_por_tercis(indice):
    """Corta o índice em 3 faixas com quantidades iguais de cada classe."""
    return pd.qcut(indice, q=3, labels=["Eficiente", "Moderado", "Ineficiente"])

def main():
    df = pd.read_csv(CAMINHO_ENTRADA)
    df[COLUNA_BOOLEANA] = df[COLUNA_BOOLEANA].astype(int)

    indice = calcular_indice_ineficiencia(df)
    df["categoria"] = rotular_por_tercis(indice)
    df.to_csv(CAMINHO_SAIDA, index=False)
    print("Distribuição das categorias:")
    print(df["categoria"].value_counts(), "\n")

if __name__ == "__main__":
    main()

Distribuição das categorias:
categoria
Eficiente      1667
Ineficiente    1667
Moderado       1666
Name: count, dtype: int64 



In [6]:
df = pd.read_csv("/content/base_energetica_rotulada.csv")
df.head()

,id_cliente,consumo_kwh,uso_horario_pico,quantidade_equipamentos,tipo_imovel,horas_alto_consumo,categoria
0,1,524.0,0,21,Apartamento,4.7,Moderado
1,2,851.3,0,20,Comercial,8.3,Moderado
2,3,598.5,1,1,Comercial,0.7,Eficiente
3,4,790.5,1,21,Comercial,6.2,Ineficiente
4,5,673.0,0,10,Comercial,5.8,Eficiente


In [7]:
df['estimativa_financeira'] = df['consumo_kwh'] * TARIFA_KWH
df['consumo_kwh'] = df['consumo_kwh'].round(1)
df.head()

,id_cliente,consumo_kwh,uso_horario_pico,quantidade_equipamentos,tipo_imovel,horas_alto_consumo,categoria,estimativa_financeira
0,1,524.0,0,21,Apartamento,4.7,Moderado,393.000
1,2,851.3,0,20,Comercial,8.3,Moderado,638.475
2,3,598.5,1,1,Comercial,0.7,Eficiente,448.875
3,4,790.5,1,21,Comercial,6.2,Ineficiente,592.875
4,5,673.0,0,10,Comercial,5.8,Eficiente,504.750


## 4. Importação do modelo para as recomendações

In [8]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print(f"Modelo carregado em: {model.device}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Modelo carregado em: cpu


In [9]:
# Função geradora de recomendações
"""
#
"""
def gerar_recomendacoes(df: dict, categoria: str, max_new_tokens: int = 200) -> list[str]:
    """
    Gera 3 recomendações de eficiência energética com base nos dados de entrada
    e na categoria já calculada pelo classificador (Random Forest).
    """
    system_prompt = (
        "Você é um assistente especializado em eficiência energética. "
        "Responda sempre em português do Brasil, de forma objetiva."
    )

    user_prompt = f"""Com base nos dados abaixo, gere exatamente 3 recomendações curtas
e práticas para melhorar a eficiência energética do imóvel. ATENÇÃO: Não sugira recomendações com base em informações
que você NÃO conhece, como uso de ar-condicionado, por exemplo.

Dados do imóvel:
- Consumo mensal: {df['consumo_kwh']} kWh
- Uso em horário de pico: {"Sim" if df['uso_horario_pico'] else "Não"}
- Quantidade de equipamentos: {df['quantidade_equipamentos']}
- Tipo de imóvel: {df['tipo_imovel']}
- Horas de alto consumo por dia: {df['horas_alto_consumo']}
- Categoria de eficiência: {categoria}

Responda APENAS com as 3 recomendações, uma por linha, sem numeração,
sem introdução e sem comentários adicionais."""

    mensagens = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    texto_prompt = tokenizer.apply_chat_template(
        mensagens, tokenize=False, add_generation_prompt=True
    )
    model_inputs = tokenizer([texto_prompt], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )

    # Remove os tokens do prompt de entrada, mantendo só a resposta gerada
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    resposta = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    recomendacoes = [
        linha.strip("-•* ").strip()
        for linha in resposta.strip().split("\n")
        if linha.strip()
    ]
    return recomendacoes[:3]

In [12]:
cliente_id = random.randint(1, 5000)
linha_cliente = df[df["id_cliente"] == cliente_id].iloc[0]
dados_cliente = linha_cliente[["consumo_kwh", "uso_horario_pico", "quantidade_equipamentos", "tipo_imovel", "horas_alto_consumo"]].to_dict()

# Necessário adicionar a parte de definir uma categoria
categoria = linha_cliente["categoria"]
recomendacoes = gerar_recomendacoes(dados_cliente, categoria)

In [13]:
print(linha_cliente)
print(recomendacoes)

id_cliente                         100
consumo_kwh                      639.8
uso_horario_pico                     1
quantidade_equipamentos             19
tipo_imovel                       Casa
horas_alto_consumo                10.4
categoria                  Ineficiente
estimativa_financeira           479.85
Name: 99, dtype: object
['1. Reduza o número de equipamentos elétricos instalados.', '2. Implemente um plano de programação da luz para minimizar o uso na alta demanda.', '3. Opte por fontes alternativas de energia no fornecimento doméstico.']


### Exemplo saída comercial:
['1. Instale painéis solares fotovoltaicos.', '2. Faça um planejamento térmico da fachada do edifício.', '3. Optimize o gerenciamento da iluminação e das temperaturas interna.']

### Exemplo saída residencial (Casa)
['1. Reduza o número de equipamentos elétricos instalados.', '2. Implemente um plano de programação da luz para minimizar o uso na alta demanda.', '3. Opte por fontes alternativas de energia no fornecimento doméstico.']



## 5. Exploração e limpeza dos dados

In [49]:
print("Dimensões:", df.shape)
print("\nTipos de dados:")
print(df.dtypes)
print("\nValores nulos por coluna:")
print(df.isnull().sum())
print("\nRegistros duplicados:", df.duplicated().sum())

print("\nInformações adicionais:")
q33 = np.percentile(df['consumo_kwh'], 33)
q66 = np.percentile(df['consumo_kwh'], 66)
min = df['consumo_kwh'].min()
max = df['consumo_kwh'].max()
print('Consumo Minimo: ', min)
print('Consumo Maximo: ', max)
print('Percentil 33: ', q33)
print('Percentil 66: ', q66)

df.head().style.hide(axis="index")

Dimensões: (5000, 6)

Tipos de dados:
id_cliente                   int64
consumo_kwh                float64
uso_horario_pico              bool
quantidade_equipamentos      int64
tipo_imovel                 object
horas_alto_consumo         float64
dtype: object

Valores nulos por coluna:
id_cliente                 0
consumo_kwh                0
uso_horario_pico           0
quantidade_equipamentos    0
tipo_imovel                0
horas_alto_consumo         0
dtype: int64

Registros duplicados: 0

Informações adicionais:
Minimo:  129.1
Maximo:  1139.0
Percentil 33:  459.5
Percentil 66:  640.934


id_cliente,consumo_kwh,uso_horario_pico,quantidade_equipamentos,tipo_imovel,horas_alto_consumo
1,752.000000,False,11,Comercial,7.900000
2,348.300000,False,6,Apartamento,4.500000
3,579.900000,True,10,Casa,10.400000
4,326.200000,False,19,Apartamento,3.300000
5,484.800000,True,16,Casa,3.700000


In [43]:
# Variáveis numéricas
print("Estatísticas descritivas")
display(df[["consumo_kwh", "quantidade_equipamentos", "horas_alto_consumo"]].describe())

# Variáveis categóricas/booleanas
print("\nDistribuição de tipo_imovel")
print(df["tipo_imovel"].value_counts())
print("\nDistribuição de uso_horario_pico")
print(df["uso_horario_pico"].value_counts(normalize=True).round(3))

Estatísticas descritivas


,consumo_kwh,quantidade_equipamentos,horas_alto_consumo
count,5000.000000,5000.000000,5000.000000
mean,564.177660,12.679200,6.123600
std,184.576865,7.177144,3.468347
min,129.100000,1.000000,0.000000
25%,420.075000,6.000000,3.200000
50%,544.050000,12.000000,6.100000
75%,703.175000,19.000000,9.200000
max,1139.000000,25.000000,12.000000



Distribuição de tipo_imovel
tipo_imovel
Comercial      1684
Casa           1679
Apartamento    1637
Name: count, dtype: int64

Distribuição de uso_horario_pico
uso_horario_pico
False    0.509
True     0.491
Name: proportion, dtype: float64
